# RAG Step 4: Passing Context to Gemini / Groq
**Example topic: Renewable Energy**

This notebook demonstrates how retrieved document chunks are placed into a prompt before an LLM answers. API keys are optional; without a key, retrieval and prompt-building still work.

## Setup — Install Libraries

In [ ]:
!pip install -q langchain-text-splitters chromadb sentence-transformers groq google-genai

## 1. Enter API Keys Safely
You may add `GROQ_API_KEY` and/or `GEMINI_API_KEY` in Colab Secrets. You can also leave them blank to run the retrieval and prompt-building parts only.

In [ ]:
import os
import getpass

def get_key(name):
    key = os.environ.get(name, "")
    if not key:
        try:
            from google.colab import userdata
            key = userdata.get(name) or ""
        except Exception:
            key = ""
    if not key:
        key = getpass.getpass(f"Enter {name} (press Enter to skip): ").strip()
    return key

GROQ_API_KEY = get_key("GROQ_API_KEY")
GEMINI_API_KEY = get_key("GEMINI_API_KEY")
print("Groq key provided:", bool(GROQ_API_KEY))
print("Gemini key provided:", bool(GEMINI_API_KEY))
if not GROQ_API_KEY and not GEMINI_API_KEY:
    print("No API keys provided. LLM calls will be skipped; retrieval will still work.")

## 2. Load, Chunk, Store, and Retrieve Documents
The knowledge base is about solar panels, wind energy, and home battery storage.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from chromadb.utils import embedding_functions

documents = {
    "renewable_energy.txt": """Renewable Energy Basics

Renewable energy comes from sources that are naturally replenished, such as sunlight, wind, and flowing water.
Solar panels convert sunlight into electrical energy using photovoltaic cells.
Wind turbines convert the movement of air into electricity by rotating blades connected to a generator.
Renewable energy can reduce dependence on fossil fuels, although production varies with weather and time of day.""",
    "solar_storage.txt": """Solar Panels and Battery Storage

Solar panels usually generate the most electricity during sunny daytime hours.
A home battery stores extra electricity produced during the day so it can be used later, such as at night.
An inverter converts direct current from panels or batteries into alternating current used by many household appliances.
Suitable system sizing, energy efficiency, and safe installation help a solar power system work effectively."""
}

splitter = RecursiveCharacterTextSplitter(
    chunk_size=250, chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " ", ""]
)
client = chromadb.Client()
embedder = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
collection = client.create_collection(
    name="renewable_energy_rag",
    embedding_function=embedder
)

for source, text in documents.items():
    chunks = splitter.split_text(text)
    collection.add(
        documents=chunks,
        metadatas=[{"source": source, "chunk_index": i} for i in range(len(chunks))],
        ids=[f"{source}_chunk_{i}" for i in range(len(chunks))]
    )

print("Total chunks stored:", collection.count())

In [ ]:
def retrieve_context(query, n_results=4, similarity_threshold=0.25):
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )
    retrieved = []
    for doc, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        similarity = 1 - distance
        if similarity >= similarity_threshold:
            retrieved.append({
                "text": doc,
                "source": meta["source"],
                "chunk_index": meta["chunk_index"],
                "similarity": round(similarity, 4)
            })
    return retrieved

test_chunks = retrieve_context("How can a home use solar power at night?")
print("Chunks retrieved:", len(test_chunks))
for item in test_chunks:
    print(f"[{item['similarity']}] {item['source']} chunk {item['chunk_index']}")

## 3. Build a Grounded Prompt
The system prompt instructs the model to answer only from retrieved context and to admit when the documents do not contain an answer.

In [ ]:
SYSTEM_PROMPT = (
    "Answer using ONLY the supplied context. "
    "If the answer is not in the context, reply exactly: "
    "\"I don't know based on the provided documents.\" "
    "Keep the answer short and mention the source file names."
)

def build_user_prompt(query, context_chunks):
    context_text = "\n\n".join(
        f"[Source: {item['source']}, chunk {item['chunk_index']}]\n{item['text']}"
        for item in context_chunks
    )
    return f"Context:\n{context_text}\n\nQuestion: {query}\n\nAnswer:"

sample_query = "How can a home use solar power at night?"
sample_chunks = retrieve_context(sample_query)
print(build_user_prompt(sample_query, sample_chunks))

## 4. Call Groq (Optional)

In [ ]:
from groq import Groq

GROQ_MODELS = ["llama-3.3-70b-versatile", "llama-3.1-8b-instant"]

def ask_groq(user_prompt):
    if not GROQ_API_KEY:
        return "[Groq skipped: no API key provided]"
    client_groq = Groq(api_key=GROQ_API_KEY)
    last_error = None
    for model_name in GROQ_MODELS:
        try:
            response = client_groq.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.1,
                max_tokens=500
            )
            if response.choices[0].message.content:
                return response.choices[0].message.content.strip()
        except Exception as error:
            last_error = error
    return f"[Groq error: {last_error}]"

print(ask_groq(build_user_prompt(sample_query, sample_chunks)))

## 5. Call Gemini (Optional)

In [ ]:
from google import genai
from google.genai import types

GEMINI_MODELS = ["gemini-2.5-flash", "gemini-2.5-flash-lite"]

def ask_gemini(user_prompt):
    if not GEMINI_API_KEY:
        return "[Gemini skipped: no API key provided]"
    gemini_client = genai.Client(api_key=GEMINI_API_KEY)
    last_error = None
    for model_name in GEMINI_MODELS:
        try:
            response = gemini_client.models.generate_content(
                model=model_name,
                contents=user_prompt,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    temperature=0.1,
                    max_output_tokens=800
                )
            )
            if response.text:
                return response.text.strip()
        except Exception as error:
            last_error = error
    return f"[Gemini error: {last_error}]"

print(ask_gemini(build_user_prompt(sample_query, sample_chunks)))

## 6. Full RAG Function and Tests

In [ ]:
def rag_answer(query, provider="groq", show_sources=True):
    chunks = retrieve_context(query)
    if not chunks:
        return "I don't know based on the provided documents. (No relevant chunks were retrieved.)"
    prompt = build_user_prompt(query, chunks)
    answer = ask_gemini(prompt) if provider == "gemini" else ask_groq(prompt)
    if show_sources:
        sources = sorted({f"{c['source']} #{c['chunk_index']}" for c in chunks})
        answer += "\n\nRetrieved from: " + ", ".join(sources)
    return answer

default_provider = "groq" if GROQ_API_KEY else "gemini"
print("Default provider:", default_provider)

q1 = "What do solar panels do?"
print("\nQuestion:", q1)
print(rag_answer(q1, provider=default_provider))

q2 = "Why is a home battery useful?"
print("\nQuestion:", q2)
print(rag_answer(q2, provider=default_provider))

q3 = "Who won the 2024 football final?"
print("\nQuestion:", q3)
print(rag_answer(q3, provider=default_provider))

## Key Takeaways
- Documents are split into chunks and stored in ChromaDB.
- A query retrieves relevant chunks before the LLM is called.
- The prompt contains both context and the question.
- API keys are optional for testing retrieval and prompt construction.
- Out-of-scope questions should receive an honest fallback instead of a guessed answer.